[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C25_Long_Context_Course/02_flash_attention/02_flash_attention.ipynb)

# 02 · FlashAttention 与 IO 复杂度（用 numpy 从零写出）

**内存墙**：朴素注意力要 materialize 一个 `n×n` 矩阵，128k 单头就要 ~34GB。
FlashAttention 用 **online softmax + 分块**永不存这个矩阵，显存 O(n²)→O(n)，且与朴素**逐位相同**。

**路线**：
1. 朴素注意力（参考实现，造出完整 n×n 的 S）
2. online softmax + 输出累加器：两块合并
3. **分块 FlashAttention 前向** → 对拍朴素，`atol=1e-10`
4. 显存账：O(n²) vs O(n)
5. IO/HBM 账：为什么更快（FLOPs 反而略多）
6. LSE 与重算：反向省显存的钥匙
7. ✏️ 练习（online softmax 合并 / blockwise 累积 / IO 计数 / 数值稳定）→ 📖 答案 → 🧪 胶囊

> 本模块与 C36 互补：C36 讲 GPU 内核（thread/tiling），这里讲它在**长上下文**中的系统角色与 IO 账。

## 1 · 朴素注意力（参考实现）

$O=\mathrm{softmax}(QK^\top/\sqrt{d})V$。朴素实现把整个 $n\times n$ 的 S、P **materialize 出来**——长序列爆显存的根源。它作为对拍的 **ground truth**。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def softmax_rows(S):
    m = S.max(axis=-1, keepdims=True)
    e = np.exp(S - m)
    return e / e.sum(axis=-1, keepdims=True)

def attention_naive(Q, K, V, causal=False):
    n, d = Q.shape
    S = Q @ K.T / np.sqrt(d)              # (n,n) 完整分数矩阵 O(n^2) 显存
    if causal:
        S = np.where(np.triu(np.ones((n, n), bool), k=1), -np.inf, S)
    P = softmax_rows(S)                   # (n,n) 完整概率矩阵
    return P @ V

n, d = 6, 4
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
O_ref = attention_naive(Q, K, V)
print('朴素注意力输出形状:', O_ref.shape)
S = Q @ K.T / np.sqrt(d)
assert np.allclose(softmax_rows(S).sum(axis=1), 1.0)
print('softmax 每行和=1 ✅（但我们刚 materialize 了整个', S.shape, '矩阵）')

## 2 · online softmax + 输出累加器：两块合并

online softmax 流式维护 `m`(运行最大值)、`l`(运行指数和)。FlashAttention 多维护 **`O`(运行输出累加器)**。
**唯一新东西**：最大值刷新时，`l` 和 `O` 都要乘校正因子 `corr=exp(m_old-m_new)` 回缩（`O` 里已累加项以旧 `m` 为基准）。
先写「合并一个新块」，再验证：把 K/V 切两半分别合并 == 朴素。

In [ ]:
def merge_block(O, l, m, Sblock, Vblock):
    '''把一个新 KV 块合并进运行状态 (O,l,m)。Sblock:(n,bk) 已缩放分数; Vblock:(bk,d)。'''
    m_block = Sblock.max(axis=1, keepdims=True)
    m_new   = np.maximum(m, m_block)
    P       = np.exp(Sblock - m_new)                 # 对齐到新基准
    corr    = np.exp(m - m_new)                      # 校正因子（m=-inf 时 corr=0）
    l = corr * l + P.sum(axis=1, keepdims=True)
    O = corr * O + P @ Vblock
    return O, l, m_new

n, d = 12, 4
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
scale = 1.0/np.sqrt(d)
O_ref = attention_naive(Q, K, V)
O = np.zeros((n, d)); l = np.zeros((n, 1)); m = np.full((n, 1), -np.inf)
for j0 in [0, 6]:
    Sb = (Q @ K[j0:j0+6].T) * scale
    O, l, m = merge_block(O, l, m, Sb, V[j0:j0+6])
out = O / l
print('两块合并 vs 朴素 最大误差:', np.abs(out - O_ref).max())
assert np.allclose(out, O_ref, atol=1e-10)
print('✅ online softmax + 输出累加器：分两块流式合并 == 朴素一次算完')

## 3 · 分块 FlashAttention 前向

把第 2 节的合并放进循环：**query 全程在外（FA-2 风格），K/V 块在内层流式扫**。运行状态一直留着，**n×n 分数矩阵从未存在**。

In [ ]:
def flash_attention(Q, K, V, block_kv=16):
    n, d = Q.shape; scale = 1.0/np.sqrt(d)
    O = np.zeros((n, d)); m = np.full((n, 1), -np.inf); l = np.zeros((n, 1))
    for j0 in range(0, n, block_kv):
        Kj = K[j0:j0+block_kv]; Vj = V[j0:j0+block_kv]
        Sij = (Q @ Kj.T) * scale            # (n,bk) 小分数块，绝非 n×n
        m_new = np.maximum(m, Sij.max(axis=1, keepdims=True))
        P = np.exp(Sij - m_new)
        corr = np.exp(m - m_new)
        l = corr * l + P.sum(axis=1, keepdims=True)
        O = corr * O + P @ Vj
        m = m_new
    return O / l

n, d = 20, 8
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
O_ref = attention_naive(Q, K, V)
for bk in [1, 7, 8, 16, n]:                 # 含不整除 n 的 7
    out = flash_attention(Q, K, V, block_kv=bk)
    err = np.abs(out - O_ref).max()
    assert np.allclose(out, O_ref, atol=1e-10), (bk, err)
    print(f'block_kv={bk:>2d}: 与朴素最大误差 {err:.2e} ✅')
print('\n关键：不论块宽怎么切，结果都与朴素逐位相同 —— 分块没有引入近似（望远镜相消）。')

## 4 · 显存账：O(n²) vs O(n)

朴素被 $n\times n$ 的 S 支配 → $\Theta(n^2)$；Flash 只需 O、m、l、当前块 → 在 $n^2$ 量级是 $\Theta(n)$。这就是长上下文显存可行的关键。

In [ ]:
def mem_naive(n, d, b=2):  return b * (n*n + n*d)             # S 支配
def mem_flash(n, d, bk, b=2): return b * (n*d + n + bk*d)     # O + m,l + 当前块

print(f"{'n':>8s} {'朴素(GB)':>12s} {'Flash(GB)':>12s} {'倍数':>10s}")
for n in [4096, 32768, 131072]:
    mn, mf = mem_naive(n, 128), mem_flash(n, 128, 128)
    print(f'{n:>8d} {mn/1e9:>12.3f} {mf/1e9:>12.5f} {mn/mf:>9.0f}x')
assert mem_flash(131072, 128, 128) < mem_naive(131072, 128)
print('\n✅ n 越大差距越夸张：朴素随 n 平方膨胀（128k 单头 34GB），Flash 随 n 线性 —— 长上下文的钥匙。')

## 5 · IO/HBM 账：为什么更快

FlashAttention 的 FLOPs 并不比朴素少（甚至略多），它快是因为 **HBM 访问少一个数量级**。
朴素要把 $n\times n$ 的 S、P 写出再读回（$\Theta(n^2)$）；Flash 只读 Q,K,V、写 O（$\Theta(nd)$ 级）。

In [ ]:
def hbm_naive(n, d, b=2):
    # 读 Q,K,V(3nd) + 写S(n^2) + 读S写P(2n^2) + 读P(n^2) + 写O(nd)
    return b * (3*n*d + 4*n*n + n*d)
def hbm_flash(n, d, b=2):
    # 读 Q,K,V 各一次(分块复用) + 写 O；n^2 量级从不落 HBM
    return b * (3*n*d + n*d)
def flops_attn(n, d):
    return 2 * 2 * n*n*d                  # QK^T + P@V（两者各 ~2n^2d）

print(f"{'n':>8s} {'朴素HBM(GB)':>13s} {'FlashHBM(GB)':>13s} {'IO倍数':>8s} {'FLOPs相同?':>10s}")
for n in [4096, 32768, 131072]:
    hn, hf = hbm_naive(n, 128), hbm_flash(n, 128)
    print(f'{n:>8d} {hn/1e9:>13.3f} {hf/1e9:>13.4f} {hn/hf:>7.0f}x {"是":>9s}')
assert hbm_flash(32768, 128) < hbm_naive(32768, 128)
assert flops_attn(1024, 64) == flops_attn(1024, 64)   # FLOPs 不随实现变
print('\n✅ FLOPs 几乎一样，HBM 往返差一个数量级 —— 访存受限算子上，省 IO 远比省 FLOPs 值钱。')

## 6 · LSE 与重算：反向省显存的钥匙

反向需要概率 P，但前向没存 P。解法：前向只存每行的 **LSE = log-sum-exp = m + log(l)**（每行一个标量，$\Theta(n)$），
反向时用 Q,K,V + LSE **即时重算** P（用完即弃）。验证：用 LSE 重算的 P 与朴素的 P 一致。

In [ ]:
def flash_with_lse(Q, K, V, block_kv=16):
    '''flash 前向，额外返回每行 LSE（反向重算用）。'''
    n, d = Q.shape; scale = 1.0/np.sqrt(d)
    O = np.zeros((n, d)); m = np.full((n,1), -np.inf); l = np.zeros((n,1))
    for j0 in range(0, n, block_kv):
        Sij = (Q @ K[j0:j0+block_kv].T) * scale
        m_new = np.maximum(m, Sij.max(axis=1, keepdims=True))
        P = np.exp(Sij - m_new); corr = np.exp(m - m_new)
        l = corr*l + P.sum(axis=1, keepdims=True); O = corr*O + P @ V[j0:j0+block_kv]; m = m_new
    lse = m + np.log(l)                   # (n,1) 每行一个标量
    return O / l, lse

n, d = 16, 8
Q = rng.standard_normal((n, d)); K = rng.standard_normal((n, d)); V = rng.standard_normal((n, d))
O_flash, lse = flash_with_lse(Q, K, V)
# 反向重算 P：P[i,j] = exp(S[i,j] - lse[i])，无需存完整 P
S = Q @ K.T / np.sqrt(d)
P_recomputed = np.exp(S - lse)            # 用 LSE 重算
P_naive = softmax_rows(S)
assert np.allclose(P_recomputed, P_naive, atol=1e-10), 'LSE 重算的 P 应等于朴素 P'
assert lse.shape == (n, 1), 'LSE 每行只一个标量 → O(n) 显存'
print('✅ 只存 O(n) 的 LSE 就能在反向精确重算 O(n²) 的 P —— gradient checkpointing 的思想')

---
## ✏️ 练习 1：实现 online softmax 的单块合并

实现 `merge(O, l, m, S_blk, V_blk)`：把一个新 KV 块合并进运行状态。这是 FlashAttention 的核心递推。
递推：`m_new=max(m, S_blk行最大)`；`P=exp(S_blk-m_new)`；`corr=exp(m-m_new)`；`l=corr*l+sum(P)`；`O=corr*O+P@V_blk`。

In [ ]:
def merge(O, l, m, S_blk, V_blk):
    # TODO: 实现上述递推，返回 (O_new, l_new, m_new)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
n, d = 10, 4; scale = 1.0/np.sqrt(d)
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
ref = attention_naive(Q, K, V)
O = np.zeros((n,d)); l = np.zeros((n,1)); m = np.full((n,1), -np.inf)
for j0 in range(0, n, 3):
    Sb = (Q @ K[j0:j0+3].T) * scale
    O, l, m = merge(O, l, m, Sb, V[j0:j0+3])
assert np.allclose(O/l, ref, atol=1e-10), np.abs(O/l - ref).max()
print('✅ 练习 1 通过：你的单块合并逐块累积 == 朴素注意力')

## ✏️ 练习 2：用你的 merge 写出完整 blockwise 前向

用练习 1 的 `merge` 拼出 `my_flash(Q,K,V,block_kv)`：query 在外、K/V 块在内、最后归一化。
目标：任意块宽都与 `attention_naive` 逐位相同。

In [ ]:
def my_flash(Q, K, V, block_kv=8):
    n, d = Q.shape; scale = 1.0/np.sqrt(d)
    O = np.zeros((n,d)); l = np.zeros((n,1)); m = np.full((n,1), -np.inf)
    # TODO: for j0 in range(0,n,block_kv): 算 Sij=(Q@K块.T)*scale, 用 merge 更新, 最后 return O/l
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
for (n, d) in [(12, 4), (20, 8)]:
    Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
    ref = attention_naive(Q, K, V)
    for bk in [1, 5, n]:
        assert np.allclose(my_flash(Q,K,V,bk), ref, atol=1e-10), f'n={n} bk={bk}'
print('✅ 练习 2 通过：完整 blockwise 前向（任意块宽）== 朴素')

## ✏️ 练习 3：数 IO——FlashAttention 省了多少 HBM 往返？

实现 `io_ratio(n, d, b=2)`：返回「朴素 HBM 字节 ÷ Flash HBM 字节」。用第 5 节的公式。
验证：随 n 增大该比值增大（n² 项 vs nd 项），体现长上下文下 Flash 优势越大。

In [ ]:
def io_ratio(n, d, b=2):
    # TODO: 朴素 = b*(3nd + 4n^2 + nd); Flash = b*(3nd + nd); 返回 朴素/Flash
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
r_small = io_ratio(1024, 128)
r_large = io_ratio(131072, 128)
assert r_large > r_small, '序列越长，Flash 的 IO 优势越大'
assert r_small > 1.0, 'Flash 的 HBM 访问应少于朴素'
print(f'IO 倍数：n=1k 时 {r_small:.1f}x，n=128k 时 {r_large:.0f}x —— 长上下文下优势暴涨')
print('✅ 练习 3 通过：量化了「为什么 FlashAttention 对长序列尤其值钱」')

## ✏️ 练习 4：数值稳定——为什么必须减最大值

长上下文里 logit 可能很大，朴素 `exp(S)` 会溢出。实现 `safe_softmax_sum(s)`：用「减最大值」稳定地算 $\sum_j e^{s_j}$ 对应的归一化结果，
对拍朴素 `exp(s)/sum(exp(s))`，且在大 logit 下不出 nan/inf。

In [ ]:
def safe_softmax_sum(s):
    # s: 1D 分数。TODO: 减 max 再 exp 归一化，返回概率向量（数值稳定）
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
s_normal = rng.standard_normal(8)
p_ref = np.exp(s_normal)/np.exp(s_normal).sum()
assert np.allclose(safe_softmax_sum(s_normal), p_ref, atol=1e-12)
s_big = np.array([900.0, 901.0, 902.0])          # 朴素 exp 会 inf
out = safe_softmax_sum(s_big)
assert not np.isnan(out).any() and not np.isinf(out).any(), '稳定版不应 nan/inf'
assert abs(out.sum() - 1.0) < 1e-9
print('大 logit 稳定输出:', np.round(out, 4))
print('✅ 练习 4 通过：减最大值 = FlashAttention 天生数值稳定的根据')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def merge(O, l, m, S_blk, V_blk):
    m_new = np.maximum(m, S_blk.max(axis=1, keepdims=True))
    P = np.exp(S_blk - m_new)
    corr = np.exp(m - m_new)
    l = corr * l + P.sum(axis=1, keepdims=True)
    O = corr * O + P @ V_blk
    return O, l, m_new

In [ ]:
# 练习 2 参考答案
def my_flash(Q, K, V, block_kv=8):
    n, d = Q.shape; scale = 1.0/np.sqrt(d)
    O = np.zeros((n,d)); l = np.zeros((n,1)); m = np.full((n,1), -np.inf)
    for j0 in range(0, n, block_kv):
        Sij = (Q @ K[j0:j0+block_kv].T) * scale
        O, l, m = merge(O, l, m, Sij, V[j0:j0+block_kv])
    return O / l

In [ ]:
# 练习 3 参考答案
def io_ratio(n, d, b=2):
    naive = b * (3*n*d + 4*n*n + n*d)
    flash = b * (3*n*d + n*d)
    return naive / flash

In [ ]:
# 练习 4 参考答案
def safe_softmax_sum(s):
    m = s.max()
    e = np.exp(s - m)
    return e / e.sum()

---
## 🧪 真实数据胶囊：FlashAttention 解锁了哪些上下文长度？

用真实配置算账：**单层全部注意力头**的 n×n 分数矩阵有多大（Llama-7B 是 32 头）？对比 H100 的 80GB，
看朴素注意力在什么长度就**装不下**——而 FlashAttention 显存随 n 线性、根本不存这些矩阵。

In [ ]:
def layer_score_gb(n, n_heads=32, b=2):
    # TODO: 返回 单层 n_heads 个 n×n 分数矩阵合计大小(GB)。b=每元素字节(FP16=2)
    raise NotImplementedError

In [ ]:
# 自测
H100_GB = 80; N_HEADS = 32
print(f"{'序列长 n':>10s} {'单层 S(GB)':>13s} {'装得进 H100?':>16s}")
for n in [4096, 32768, 131072]:
    gb = layer_score_gb(n)
    fits = 'yes' if gb < H100_GB else 'NO (爆显存)'
    print(f'{n:>10d} {gb:>13.1f} {fits:>16s}')
assert layer_score_gb(32768) > 60, '32k×32头 应 ~69GB，逼近 H100'
assert layer_score_gb(131072) > H100_GB, '128k×32头 远超 H100'
print('\n✅ 32k 上下文时仅一层的 32 个分数矩阵就 ~69GB 逼近 H100；128k 时一层就 ~1100GB。')
print('   FlashAttention 永不存这些矩阵（显存 O(n)）→ 这就是长上下文从「不可行」到「可行」的转折。')

In [ ]:
# 📖 胶囊参考答案
def layer_score_gb(n, n_heads=32, b=2):
    return n_heads * b * n * n / 1e9

### 小结
- **内存墙**：朴素注意力 materialize n×n 的 S/P，128k 单头 ~34GB → 长上下文物理上不可行。
- **FlashAttention** = 分块 + online softmax + 输出累加器；唯一新东西是用 `corr` 同时缩放 `l` 和 `O`。
- **精确**（非近似）：校正因子望远镜相消，每项对齐到全局最大值 → 与朴素逐位相同（`atol=1e-10`）。
- **显存 O(n²)→O(n)、HBM 访问 O(n²)→O(n²d²/M)**；FLOPs 反而略多 —— 访存受限算子上省 IO 才值钱。
- **LSE + 重算**：前向只存 O(n) 的 LSE，反向即时重算 O(n²) 的 P，省显存。**FlashDecoding** 沿 KV 切段并合并状态，解长解码并行度之困。

下一站：**模块 03 · 稀疏与滑窗注意力** —— FlashAttention 让 O(n²) 算得起，但 1M 上仍然贵；下一步是把 O(n²) 直接降成 O(n·W)。